In [1]:
import parcels
import copernicusmarine
import xarray as xr
import numpy as np
import gsw

import matplotlib.pyplot as plt

In [ ]:
copernicus_args = {
    "start_datetime": "2025-06-10",
    "end_datetime": "2025-06-12",
    "minimum_longitude": -49,
    "maximum_longitude": -42,
    "minimum_latitude": 18,
    "maximum_latitude": 26,
    "service": "arco-geo-series",
    "chunk_size_limit": 1,
}

ds_uv = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m",
    variables=["uo", "vo"],
    **copernicus_args,
)

ds_w = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy-wcur_anfc_0.083deg_P1D-m",
    variables=["wo"],
    **copernicus_args,
)

ds_s = copernicusmarine.open_dataset(
  dataset_id="cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m",
  variables=["so"],
  **copernicus_args,
)

ds_t = copernicusmarine.open_dataset(
  dataset_id="cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m",
  variables=["thetao"],
  **copernicus_args,
)

# combine into one dataset
ds = parcels.convert.copernicusmarine_to_sgrid(
    fields={
        "U": ds_uv["uo"],
        "V": ds_uv["vo"],
        "W": ds_w["wo"],
        "S": ds_s["so"],
        "T": ds_t["thetao"],
    }
)

# explicitly load dataset
ds.load()
display(ds)

# convert to parcels fieldset
fieldset = parcels.FieldSet.from_sgrid_conventions(ds)
fieldset.to_chunk_cached_arrays()
# fieldset.describe()  # DEBUG very slow

In [ ]:
    # Fan and Tsuchiya, 1990
    c = 1.4 # sea water
    d = 1.4 # varies between .8 and 1.6 (dirty to clean)
    r = 1 # cm
    mu_b = 0.001*100
    sigma = 72
    M = g*100*1000*mu_b**4/(rho_f*sigma**3)
    #particles.risevel = ((rho_f/1000)*g*100*r**2*3.68*M**(-0.038)*mu_b**(-d) + c*sigma*(rho_f/1000)*r + g*100*r**(-d/2 - 1/d))/100

In [ ]:
mu = 0.001 # Pa s
L = 1e-2 # m
rho_f = 1028
rho_b = 10
g = 9.81
Re = 0.01*rho_f*L/mu # Reynolds number
CD = 24/Re + 3/Re**0.5 + 0.34 # drag coefficient
d = 1e-3 # 1 mm size of bubble
rise = (4*d*g*(1 - rho_b/rho_f)/(3*CD))**0.5
print(rise)

In [ ]:
H2Particle = parcels.Particle.add_variable(
    [
        parcels.Variable("temp", dtype=np.float32, initial=np.nan),
        parcels.Variable("salt", dtype=np.float32, initial=np.nan),
        parcels.Variable("density", dtype=np.float32, initial=np.nan),
        parcels.Variable("risevel", dtype=np.float32, initial=0.01),
    ]
)

# This is where the science happens - how to define rise velocity?
def Rising(particles, fieldset):
    # First term
    g = 9.81  # Acceleration due to gravity in m/s^2
    rho_b = 10  # bubble density
    salt = fieldset.S[particles]
    temp = fieldset.T[particles]
    pressure = gsw.p_from_z(-particles.z, particles.y)
    rho_f = gsw.density.rho(salt, temp, pressure)
    #term1 = g * (rho_b - rho_f) / rho_b

    # second term
    w = fieldset.W[particles]
    FD = 18 * mu /(rho_b * d**2) * CD * Re /24. * (w - particles.risevel)

    #particles.risevel = term1  #+ FD  # Rise velocity in m/s
    #print(rho_f, rho_b,particles.risevel)

    # simple model, for small bubbles < 2.6 mm (McGinnis et al., 2006)
    mu = 0.001 # Pa s
    L = 1 # m
    Re = particles.risevel*rho_f*L/mu # Reynolds number
    CD = 24/Re + 3/Re**0.5 + 0.34 # drag coefficient
    d = 1e-3 # 1 mm size of bubble
    particles.risevel = (4*d*g*(1 - rho_b/rho_f)/(3*CD))**0.5
    
    particles.dz -= particles.risevel * particles.dt

def DeleteAtSurface(particles, fieldset):
    through_surface = particles.z <= 0
    particles[through_surface].state = parcels.StatusCode.Delete


In [ ]:
pset = parcels.ParticleSet(fieldset, pclass=H2Particle, x=-46.15, y=22.84, z=2000)
oufile = parcels.ParticleFile(
    "output.parquet",
    outputdt=np.timedelta64(15, "m"),
    mode="w",
)

pset.execute(
    [parcels.kernels.AdvectionRK2_3D, Rising, DeleteAtSurface],
    dt=np.timedelta64(15, "m"),
    runtime=np.timedelta64(3, "D"),
    output_file=oufile,
)

In [ ]:
df = parcels.read_particlefile("output.parquet")

plt.plot(df["t"], df["z"], '.-')
plt.show()

plt.plot(df["x"], df["y"], '.-')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(13, 8))
ax = plt.axes(projection="3d")
ax.view_init(azim=-145)
ax.plot3D(df["x"], df["y"], df["z"], '.-', color="gray")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_zlabel("Depth (m)")
ax.set_zlim(df["z"].max(), 0)


In [ ]:
plt.plot(np.diff(df["z"]))
plt.show()